# Measurements of Pupil Size during Saccadic and Smooth Pursuit Eye Movements in Human Participants and Artificial Eyes Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.4drb-j8h7/fair2.json
```

*Dataset citation:* Pethe, S. and Ray, S. 2026. Measurements of Pupil Size during Saccadic and Smooth Pursuit Eye Movements in Human Participants and Artificial Eyes. Frontiers.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.4drb-j8h7/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print('Dataset identifier:', metadata.identifier)
print('Published:', metadata.datePublished)
print('Keywords:', metadata.keywords)

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we inspect all available record sets and their fields. Croissant schemas represent record sets as entities containing fields and columns, each referenced by their `@id`.

In [ ]:
# List all record sets and their fields, referencing by `@id`

record_sets = dataset.record_sets

if len(record_sets) == 0:
    print("No record sets found in the metadata.")
else:
    print(f"Found {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs['name']}")
        # Fields/columns
        if 'field' in rs:
            print("  Fields:")
            for f in rs['field']:
                print(f"    - Field @id: {f['@id']} | Name: {f.get('name', 'N/A')}")
        if 'column' in rs:
            print("  Columns:")
            for c in rs['column']:
                print(f"    - Column @id: {c['@id']} | Name: {c.get('name', 'N/A')}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract and preview records from all available record sets, referencing each by its `@id`.

In [ ]:
# Collect all record set @id's
record_sets = dataset.record_sets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nRecordSet @id: {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Could not load records for RecordSet @id {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, we'll select a numeric field from one of the DataFrames (record set), filter for values above a threshold, normalize it, and group by a categorical attribute.

In [ ]:
# Example: Analyze numeric columns from the first record set

if len(dataframes) > 0:
    first_rs_id = record_set_ids[0]
    df = dataframes[first_rs_id]
    print(f"Analyzing RecordSet @id: {first_rs_id}")
    
    # Try to find a numeric field
    numeric_col = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_col = col
            break

    if numeric_col is not None:
        threshold = df[numeric_col].mean() if not pd.isnull(df[numeric_col]).all() else 10
        filtered_df = df[df[numeric_col] > threshold]
        print(f"Filtered records with {numeric_col} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
        print(f"Normalized {numeric_col} for filtered records:")
        print(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())

        # Try grouping by a categorical field
        group_col = None
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and col!=numeric_col:
                group_col = col
                break

        if group_col is not None:
            grouped_df = filtered_df.groupby(group_col).mean(numeric_only=True)
            print(f"Grouped data by {group_col}:")
            print(grouped_df.head())
        else:
            print("No categorical grouping column found.")
    else:
        print("No numeric fields found to analyze.")
else:
    print("No dataframes loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot distributions for numeric fields (if present) and visualize relationships between record fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric column distribution
if len(dataframes) > 0 and numeric_col is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_col].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_col} in RecordSet @id {first_rs_id}")
    plt.xlabel(numeric_col)
    plt.ylabel("Frequency")
    plt.show()
    
    # If grouped_df exists, plot group means
    if 'grouped_df' in locals() and grouped_df.shape[0] > 0:
        grouped_df[numeric_col].plot(kind='bar')
        plt.title(f"Mean {numeric_col} by {group_col} in filtered data")
        plt.xlabel(group_col)
        plt.ylabel(f"Mean {numeric_col}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded FAIR^2 dataset metadata and explored available record sets and fields by their `@id`.
- Data extraction used the `mlcroissant` library to load and preview table data.
- Simple EDA steps demonstrated filtering and normalization on numeric columns, as well as grouping by a categorical field.
- Visualizations showed basic distributions and group summaries.

This notebook serves as a starting point for FAIR analysis, benchmark studies, and advanced computational investigations using Croissant-compliant datasets.